In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

**Tiny Input**

In [3]:
batch_size = 1
seq_len = 4
d_model = 8

x = torch.rand(batch_size, seq_len, d_model)
print("Input shape :", x.shape)
print(x)

Input shape : torch.Size([1, 4, 8])
tensor([[[0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
         [0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895],
         [0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071],
         [0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278]]])


**Single scaled dot-product attention function**

In [4]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    print("\n--- Inside scaled_dot_product_attention ---")
    print("Q shape:", Q.shape)
    print("K shape:", K.shape)
    print("V shape:", V.shape)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    print("\nAttention scores shape:", scores.shape)
    print(scores)

    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
        print("\nMasked scores:")
        print(scores)

    attention_weights = torch.softmax(scores, dim=-1)
    print("\nAttention weights shape:", attention_weights.shape)
    print(attention_weights)
    
    output = torch.matmul(attention_weights, V)
    print("\nAttention output shape:", output.shape)
    print(output)
    
    return output, attention_weights

In [5]:
Q = x
K = x
V = x

output, attention_weights = scaled_dot_product_attention(Q, K, V)


--- Inside scaled_dot_product_attention ---
Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])

Attention scores shape: torch.Size([1, 4, 4])
tensor([[[1.2267, 1.1065, 0.8914, 0.7826],
         [1.1065, 1.2752, 0.7482, 0.8980],
         [0.8914, 0.7482, 0.8689, 0.5305],
         [0.7826, 0.8980, 0.5305, 0.8248]]])

Attention weights shape: torch.Size([1, 4, 4])
tensor([[[0.3083, 0.2734, 0.2205, 0.1978],
         [0.2707, 0.3204, 0.1892, 0.2197],
         [0.2824, 0.2447, 0.2761, 0.1968],
         [0.2536, 0.2847, 0.1971, 0.2646]]])

Attention output shape: torch.Size([1, 4, 8])
tensor([[[0.7430, 0.4197, 0.4757, 0.6045, 0.5247, 0.5878, 0.6580, 0.4875],
         [0.7303, 0.4195, 0.5158, 0.6121, 0.5481, 0.5419, 0.6684, 0.4873],
         [0.7431, 0.4045, 0.4460, 0.5732, 0.4930, 0.6067, 0.6562, 0.5007],
         [0.7280, 0.4201, 0.5247, 0.5734, 0.5348, 0.5308, 0.6652, 0.4807]]])


**learnable projection**

In [6]:
W_q = nn.Linear(d_model, d_model)
W_k = nn.Linear(d_model, d_model)
W_v = nn.Linear(d_model, d_model)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

print("Projected Q shape:", Q.shape)
print("Projected K shape:", K.shape)
print("Projected V shape:", V.shape)

Projected Q shape: torch.Size([1, 4, 8])
Projected K shape: torch.Size([1, 4, 8])
Projected V shape: torch.Size([1, 4, 8])


In [7]:
output, attention_weights = scaled_dot_product_attention(Q, K, V)


--- Inside scaled_dot_product_attention ---
Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])

Attention scores shape: torch.Size([1, 4, 4])
tensor([[[ 0.0303,  0.1118,  0.1073,  0.0392],
         [-0.0447,  0.0550,  0.0634,  0.0239],
         [-0.1174,  0.0379, -0.1388, -0.0249],
         [-0.1392, -0.0172, -0.0512, -0.0112]]], grad_fn=<DivBackward0>)

Attention weights shape: torch.Size([1, 4, 4])
tensor([[[0.2396, 0.2599, 0.2588, 0.2417],
         [0.2331, 0.2575, 0.2597, 0.2497],
         [0.2356, 0.2752, 0.2307, 0.2585],
         [0.2294, 0.2592, 0.2505, 0.2608]]], grad_fn=<SoftmaxBackward0>)

Attention output shape: torch.Size([1, 4, 8])
tensor([[[ 2.3898e-01,  1.4964e-01, -6.3270e-03, -2.1874e-01, -6.6917e-01,
          -4.7470e-01,  4.7456e-01, -4.1689e-01],
         [ 2.4000e-01,  1.5005e-01, -4.2492e-03, -2.1787e-01, -6.6718e-01,
          -4.7255e-01,  4.7504e-01, -4.1581e-01],
         [ 2.4392e-01,  1.4254e-01,  5.0429e-03, -2.05